# Práctica Guiada: Web Scraping de extremo a extremo (ETL con BeautifulSoup y Pandas)

En esta práctica guiada vamos a construir un flujo completo de **Extracción, Transformación y Carga (ETL)** desde cero utilizando `requests`, `BeautifulSoup` y `pandas`.

### Objetivo del ejercicio
Extraer información de múltiples páginas/secciones de **FilmAffinity** para construir un `DataFrame` limpio con **4 variables clave por película**:
1. **Título** (`str`)
2. **Año de estreno** (`int`)
3. **Director/a** (`str`)
4. **Nota media / Rating** (`float`)

---

## Paso 1: Importación de librerías y configuración

Importamos las librerías necesarias. Añadimos `time` para hacer pausas entre peticiones (`time.sleep`) y `re` / métodos de strings para limpiar el año.

In [ ]:
import os
import time
# sirve para buscar patrones dentro de texto
import re
import requests
from bs4 import BeautifulSoup as bs
import pandas as pd

# Configuramos Pandas para que muestre textos largos sin cortarlos
pd.set_option('max_colwidth', 800)

# Definimos nuestro 'disfraz' de navegador (User-Agent) para evitar bloqueos 403
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"
}



## Paso 2: Regla del Francotirador (Inspeccionar y probar con 1 sola película)

**Buena práctica profesional:** Nunca escribas directamente un bucle para recorrer 100 elementos o varias páginas a ciegas. Primero descargamos una página, localizamos la **primera tarjeta (`cards[0]`)** y verificamos con `print()` que sabemos extraer y limpiar los 4 campos sin errores.

Abre el Inspector del navegador (`F12`) en `https://www.filmaffinity.com/es/topgen.php` y observa la estructura de cada película:
* **Contenedor padre de cada película:** `<div class="movie-card">`
* **Título y Año:** Dentro de `<div class="mc-title">` (el título está en `<a>` y el año entre paréntesis o en etiqueta anexa)
* **Director:** Dentro de `<div class="mc-director">`
* **Nota media:** Dentro de `<div class="avg">`

In [ ]:
# 1. Hacemos la petición de prueba a la primera página
url_prueba = "https://www.filmaffinity.com/es/topgen.php"
response = requests.get(url_prueba, headers=headers, timeout=10)
response.raise_for_status()  # Verificamos que responde 200 OK

# 2. Creamos la sopa
soup = bs(response.content, "html.parser")

# 3. Buscamos todas las tarjetas de películas y seleccionamos SOLO LA PRIMERA (índice 0) para probar
cards = soup.find_all("div", class_="movie-card")
print(f"Total de películas encontradas en la página: {len(cards)}")

primera_peli = cards[0]

In [ ]:
# Probamos la extracción y limpieza de los 4 datos sobre 'primera_peli'

# 1. TÍTULO: está dentro de la etiqueta <a> que cuelga de div.mc-title
caja_titulo = primera_peli.find("div", class_="mc-title")
titulo = caja_titulo.find("a").get_text()

# 2. AÑO: buscamos cualquier número de 4 dígitos (ej. 1972) dentro de la tarjeta o en mc-year / mc-title
# (Usamos una expresión regular sencilla o búsqueda de etiqueta para que sea 100% robusto)
span_anio = primera_peli.find("span", class_="mc-year")

anio = int(span_anio.get_text())


# 3. DIRECTOR: está dentro de div.mc-director
caja_director = primera_peli.find("div", class_="mc-director")

if caja_director:
    director = caja_director.get_text()
else:
    director = "Desconocido"

# 4. NOTA MEDIA: cambiamos la coma decimal por punto y convertimos a float
caja_nota = primera_peli.find("div", class_="avg")

if caja_nota:
    nota = float(caja_nota.get_text().replace(",", ".")) 
else:
    nota = None

print(f"Título:   {titulo}")
print(f"Año:      {anio}")
print(f"Director: {director}")
print(f"Nota:     {nota}")

## Paso 3: Extracción Multi-Página con Bucle y `time.sleep` (Nivel 2)

Una vez comprobado que nuestra lógica funciona para una tarjeta, vamos a automatizar la recolección recorriendo **varias páginas/rankings de FilmAffinity**.

* **Nota técnica:** En FilmAffinity podemos recorrer distintas URLs o páginas de rankings (por ejemplo, combinando el Top General `topgen.php` con rankings específicos o paginados) porque todas utilizan la misma estructura de tarjetas `div.movie-card`.
* **Regla ética y técnica:** Entre cada petición `requests.get()`, añadimos `time.sleep(2)` para esperar 2 segundos y no saturar el servidor.

In [ ]:
# Lista de URLs a scrapear (puedes añadir diferentes páginas o filtros de país/género en topgen.php)
# Por ejemplo: Top Histórico General, Top Cine Español (country=ES) y Top Cine Japonés (country=JP)
urls_a_scrapear = [
    ("Global", "https://www.filmaffinity.com/es/topgen.php"),
    ("España", "https://www.filmaffinity.com/es/topgen.php?country=ES"),
    ("Japón",  "https://www.filmaffinity.com/es/topgen.php?country=JP")
]

# Inicializamos el diccionario vacío donde acumularemos los datos de todas las páginas
datos_peliculas = {
    "Ranking_Origen": [],
    "Titulo": [],
    "Anio": [],
    "Director": [],
    "Nota": []
}

# BUCLE EXTERIOR: recorre cada página web
for origen, url in urls_a_scrapear:
    print(f"Descargando página de ranking: {origen} ({url})...")

    # Bloque fijo de BeautifulSoup
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
    soup_pagina = bs(response.content, "html.parser")


    tarjetas = soup_pagina.find_all("div", class_="movie-card")
    
    # BUCLE INTERIOR: recorre las películas de esa página (cogemos el Top 15 de cada una)
    for card in tarjetas[:15]:
        # 1. Título
        caja_titulo = card.find("div", class_="mc-title")
        titulo = caja_titulo.find("a").get_text() if (caja_titulo and caja_titulo.find("a")) else None
        
        # 2. Año
        span_anio = card.find("span", class_="mc-year")
        texto_anio = span_anio.get_text()
        anio = int(texto_anio) if texto_anio.isdigit() else None
        
            
        # 3. Director
        caja_director = card.find("div", class_="mc-director")
        director = caja_director.get_text() if caja_director else None
        
        # 4. Nota
        caja_nota = card.find("div", class_="avg")
        texto_nota = caja_nota.get_text(strip=True).replace(",", ".") if caja_nota else ""
        try:
            nota = float(texto_nota)
        except ValueError:
            nota = None
            
        # Guardamos los datos en las listas de nuestro diccionario
        datos_peliculas["Ranking_Origen"].append(origen)
        datos_peliculas["Titulo"].append(titulo)
        datos_peliculas["Anio"].append(anio)
        datos_peliculas["Director"].append(director)
        datos_peliculas["Nota"].append(nota)
        
    # PAUSA DE CORTESÍA: esperamos 2 segundos antes de pedir la siguiente página al servidor
    time.sleep(2)

print("¡Extracción completada con éxito!")

## Paso 4: Creación del DataFrame y Exploración en Pandas

Convertimos nuestro diccionario `datos_peliculas` en un `DataFrame` de Pandas y revisamos que los tipos de datos (`dtypes`) y la estructura sean correctos.

In [ ]:
# Creamos el DataFrame a partir del diccionario recolectado
df_pelis = pd.DataFrame(datos_peliculas)

# Comprobamos dimensiones y tipos de datos
print(f"Dimensiones del DataFrame: {df_pelis.shape}")
print("\nTipos de datos por columna:")
print(df_pelis.dtypes)

# Mostramos las primeras 10 filas
df_pelis.head(10)